# Test Scala Data Provenance

### Prerequisites

In [1]:
import $ivy.`org.apache.spark::spark-sql:4.1.1`

import $ivy.$

In [2]:
val version = scala.io.Source.fromFile("../VERSION")  // Get version from file
  .getLines().next().trim

interp.load.ivy("org.dataprov.dp" %% "dp-spark" % version)  // use porogrammatic API 

// // For publishLocal (~/.ivy2/local)
// import $ivy.`org.dataprov.dp::dp-spark:0.0.1` // N.B: version must be specified explicitly with this method

// // For publishM2 (~/.m2)
// import $repo.`file:///home/ronan/.m2/repository`
// import $ivy.`org.dataprov.dp::dp-spark:0.0.1` // N.B: version must be specified explicitly with this method

version: String = "0.0.1"

### Import libraries

In [3]:
import java.sql.Date

import org.apache.spark.sql.{SparkSession, DataFrame, Dataset}
import org.apache.spark.sql.functions._
import org.apache.spark.sql.execution.SparkPlan

import org.apache.spark.sql.catalyst.plans.logical._
import org.apache.spark.sql.catalyst.expressions._
import org.apache.spark.sql.catalyst.expressions.aggregate._



import java.sql.Date
import org.apache.spark.sql.{SparkSession, DataFrame, Dataset}
import org.apache.spark.sql.functions._
import org.apache.spark.sql.execution.SparkPlan
import org.apache.spark.sql.catalyst.plans.logical._
import org.apache.spark.sql.catalyst.expressions._
import org.apache.spark.sql.catalyst.expressions.aggregate._

### Initialize spark session

In [4]:
val spark = SparkSession.builder()
  .master("local[*]")
  .appName("notebook")
  .getOrCreate()

// Set log level to ERROR to reduce verbosity
spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/05 14:21:15 INFO SparkContext: Running Spark version 4.1.1
26/05/05 14:21:15 INFO SparkContext: OS info Mac OS X, 26.4.1, aarch64
26/05/05 14:21:15 INFO SparkContext: Java version 17.0.10+7
26/05/05 14:21:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/05 14:21:15 INFO ResourceUtils: ==============================================================
26/05/05 14:21:15 INFO ResourceUtils: No custom resources configured for spark.driver.
26/05/05 14:21:15 INFO ResourceUtils: ==============================================================
26/05/05 14:21:15 INFO SparkContext: Submitted application: notebook
26/05/05 14:21:15 INFO SecurityManager: Changing view acls to: mac-ABALLA16
26/05/05 14:21:15 INFO SecurityManager: Changing modify acls to: mac-ABALLA16
26/05/05 14:21:15 INFO SecurityManager: Changing view acls groups to

spark: SparkSession = org.apache.spark.sql.classic.SparkSession@7da7572b

### Create spark dataframe

We create a dataframe with duplicates to test the provenance annotation.
 
The provenance annotation will count the number of duplicates for each tuple.

In [5]:
val df: DataFrame = spark.createDataFrame(
    Seq(
        ("a", "b", "c"),
        ("a", "b", "c"),
        ("d", "b", "e"),
        ("d", "b", "e"),
        ("d", "b", "e"),
        ("d", "b", "e"),
        ("d", "b", "e"),
        ("f", "g", "e")
    )
).toDF("A", "B", "C")

df.show()

+---+---+---+
|  A|  B|  C|
+---+---+---+
|  a|  b|  c|
|  a|  b|  c|
|  d|  b|  e|
|  d|  b|  e|
|  d|  b|  e|
|  d|  b|  e|
|  d|  b|  e|
|  f|  g|  e|
+---+---+---+



df: DataFrame = [A: string, B: string ... 1 more field]

We create another table with annotations.

In this case, the annotations are natural numbers which represent the multiplicity of the tuple in the multiset. 

We can compute the provenance annotation by grouping by the columns and counting the number of duplicates for each tuple.


In [6]:
val dfWithProvenance: DataFrame = df.groupBy("A", "B", "C").count().withColumnRenamed("count", "provenance").orderBy("A", "B", "C")
dfWithProvenance.show()


+---+---+---+----------+
|  A|  B|  C|provenance|
+---+---+---+----------+
|  a|  b|  c|         2|
|  d|  b|  e|         5|
|  f|  g|  e|         1|
+---+---+---+----------+



dfWithProvenance: DataFrame = [A: string, B: string ... 2 more fields]

In [7]:
// 1. Parsed Logical Plan (Unresolved)
val parsedPlan: LogicalPlan = df.queryExecution.logical

// 2. Analyzed Logical Plan (Resolved)
// This is usually what you want when inspecting the schema and resolved columns
val analyzedPlan: LogicalPlan = df.queryExecution.analyzed

// 3. Optimized Logical Plan
// This is the plan after Spark's Catalyst optimizer has applied its rules
val optimizedPlan: LogicalPlan = df.queryExecution.optimizedPlan

// 4. Physical Plan
// This is the plan produced by the physical planning strategies, 
// but BEFORE rule-based physical optimizations (like whole-stage codegen) are applied.
val physicalPlan: SparkPlan = df.queryExecution.sparkPlan

// 5. Executed Physical Plan (Final)
// This is the absolute final plan that Spark actually runs. It includes 
// execution-specific preparations like adding Exchange nodes (shuffles) 
// and WholeStageCodegen blocks.
val executedPlan: SparkPlan = df.queryExecution.executedPlan

parsedPlan: LogicalPlan = UnresolvedSubqueryColumnAliases(
  outputColumnNames = ArraySeq("A", "B", "C"),
  child = LocalRelation(
    output = List(
      AttributeReference(
        name = "_1",
        dataType = StringType,
        nullable = true,
        metadata = {}
      ),
      AttributeReference(
        name = "_2",
        dataType = StringType,
        nullable = true,
        metadata = {}
      ),
      AttributeReference(
        name = "_3",
        dataType = StringType,
        nullable = true,
        metadata = {}
      )
    ),
    data = List(
      [a,b,c],
      [a,b,c],
      [d,b,e],
      [d,b,e],
      [d,b,e],
      [d,b,e],
      [d,b,e],
      [f,g,e]
    ),
    isStreaming = false,
    stream = None
  )
)
analyzedPlan: LogicalPlan = Project(
  projectList = List(
    Alias(
      child = AttributeReference(
        name = "_1",
        dataType = StringType,
        nullable = true,
        metadata = {}
      ),
      name = "A"
    ),
    Alias(
   

### Example $ \pi_{AB}R \bowtie \pi_{BC}R$ 

In [8]:
// Expected results:
// +---+---+---+----------+
// |  A|  B|  C|provenance|
// +---+---+---+----------+
// |  a|  b|  c|         4| 2*2
// |  a|  b|  e|        10| 2*5
// |  d|  b|  c|        10| 5*2
// |  d|  b|  e|        25| 5*5
// |  f|  g|  e|         1| 1*1
// +---+---+---+----------+

val df2: DataFrame = df.select("A", "B").join(df.select("B", "C"), "B").select("A", "B", "C").orderBy("A", "B", "C")
df2.show()

val df2WithProvenance: DataFrame = df2.groupBy("A", "B", "C").count().withColumnRenamed("count", "provenance").orderBy("A", "B", "C")
df2WithProvenance.show()


+---+---+---+
|  A|  B|  C|
+---+---+---+
|  a|  b|  c|
|  a|  b|  c|
|  a|  b|  c|
|  a|  b|  c|
|  a|  b|  e|
|  a|  b|  e|
|  a|  b|  e|
|  a|  b|  e|
|  a|  b|  e|
|  a|  b|  e|
|  a|  b|  e|
|  a|  b|  e|
|  a|  b|  e|
|  a|  b|  e|
|  d|  b|  c|
|  d|  b|  c|
|  d|  b|  c|
|  d|  b|  c|
|  d|  b|  c|
|  d|  b|  c|
+---+---+---+
only showing top 20 rows
+---+---+---+----------+
|  A|  B|  C|provenance|
+---+---+---+----------+
|  a|  b|  c|         4|
|  a|  b|  e|        10|
|  d|  b|  c|        10|
|  d|  b|  e|        25|
|  f|  g|  e|         1|
+---+---+---+----------+



df2: DataFrame = [A: string, B: string ... 1 more field]
df2WithProvenance: DataFrame = [A: string, B: string ... 2 more fields]

### Example $ \pi_{AC}R \bowtie \pi_{BC}R$ 

In [ ]:
// Expected results:
// +---+---+---+----------+
// |  A|  B|  C|provenance|
// +---+---+---+----------+
// |  a|  b|  c|         4|
// |  d|  b|  e|        25|
// |  d|  g|  e|         5|
// |  f|  b|  e|         5|
// |  f|  g|  e|         1|
// +---+---+---+----------+

val df3 : DataFrame = df
  .select("A", "C")
  .join(df.select("B", "C"), "C")
  .select("A", "B", "C")
  .orderBy("A", "B", "C")

val df3WithProvenance: DataFrame = df3
  .groupBy("A", "B", "C")
  .count()
  .withColumnRenamed("count", "provenance")
  .orderBy("A", "B", "C")

df3WithProvenance.show()

val df3WithProvenanceBis = df.select("A", "C")
  .join(df.select("B", "C"), Seq("C")) 
  .groupBy("A", "B", "C")
  .agg(count("*").alias("provenance"))
  .orderBy("A", "B", "C")

df3WithProvenanceBis.show()

// Before optimization, the plan will likely involve a large shuffle due to the join on "C".
val leftAgg = df.groupBy("A", "C").agg(count("*").alias("count_A"))
val rightAgg = df.groupBy("B", "C").agg(count("*").alias("count_B"))

// Join on "C" and multiply the counts to get the provenance
val df3WithProvenanceOptimized = leftAgg
  .join(rightAgg, Seq("C"))
  .withColumn("provenance", col("count_A") * col("count_B"))
  .select("A", "B", "C", "provenance")
  .orderBy("A", "B", "C")

df3WithProvenanceOptimized.show()


+---+---+---+----------+
|  A|  B|  C|provenance|
+---+---+---+----------+
|  a|  b|  c|         4|
|  d|  b|  e|        25|
|  d|  g|  e|         5|
|  f|  b|  e|         5|
|  f|  g|  e|         1|
+---+---+---+----------+

+---+---+---+----------+
|  A|  B|  C|provenance|
+---+---+---+----------+
|  a|  b|  c|         4|
|  d|  b|  e|        25|
|  d|  g|  e|         5|
|  f|  b|  e|         5|
|  f|  g|  e|         1|
+---+---+---+----------+

+---+---+---+----------+
|  A|  B|  C|provenance|
+---+---+---+----------+
|  a|  b|  c|         4|
|  d|  b|  e|        25|
|  d|  g|  e|         5|
|  f|  b|  e|         5|
|  f|  g|  e|         1|
+---+---+---+----------+



df3: DataFrame = [A: string, B: string ... 1 more field]
df3WithProvenance: DataFrame = [A: string, B: string ... 2 more fields]
df3WithProvenanceBis: Dataset[org.apache.spark.sql.Row] = [A: string, B: string ... 2 more fields]
leftAgg: DataFrame = [A: string, C: string ... 1 more field]
rightAgg: DataFrame = [B: string, C: string ... 1 more field]
df3WithProvenanceOptimized: Dataset[org.apache.spark.sql.Row] = [A: string, B: string ... 2 more fields]

Without provenance
```
== Analyzed Logical Plan ==
A: string, B: string, C: string
Sort [A#379 ASC NULLS FIRST, B#770 ASC NULLS FIRST, C#381 ASC NULLS FIRST], true
+- Project [A#379, B#770, C#381]
   +- Project [C#381, A#379, B#770]
      +- Join Inner, (C#381 = C#771)
         :- Project [A#379, C#381]
         :  +- Project [_1#376 AS A#379, _2#377 AS B#380, _3#378 AS C#381]
         :     +- LocalRelation [_1#376, _2#377, _3#378]
         +- Project [B#770, C#771]
            +- Project [_1#766 AS A#769, _2#767 AS B#770, _3#768 AS C#771]
               +- LocalRelation [_1#766, _2#767, _3#768]
```

With provenance
```
== Analyzed Logical Plan ==
A: string, B: string, C: string, provenance: bigint
Sort [A#379 ASC NULLS FIRST, B#770 ASC NULLS FIRST, C#381 ASC NULLS FIRST], true
+- Project [A#379, B#770, C#381, count#772L AS provenance#777L]
   +- Aggregate [A#379, B#770, C#381], [A#379, B#770, C#381, count(1) AS count#772L]
      +- Sort [A#379 ASC NULLS FIRST, B#770 ASC NULLS FIRST, C#381 ASC NULLS FIRST], true
         +- Project [A#379, B#770, C#381]
            +- Project [C#381, A#379, B#770]
               +- Join Inner, (C#381 = C#771)
                  :- Project [A#379, C#381]
                  :  +- Project [_1#376 AS A#379, _2#377 AS B#380, _3#378 AS C#381]
                  :     +- LocalRelation [_1#376, _2#377, _3#378]
                  +- Project [B#770, C#771]
                     +- Project [_1#766 AS A#769, _2#767 AS B#770, _3#768 AS C#771]
                        +- LocalRelation [_1#766, _2#767, _3#768]
```

```
== Analyzed Logical Plan ==
A: string, B: string, C: string, provenance: bigint
Sort [A#3 ASC NULLS FIRST, B#349 ASC NULLS FIRST, C#5 ASC NULLS FIRST], true
+- Project [A#3, B#349, C#5, provenance#351L]
   +- Project [C#5, A#3, count_A#335L, B#349, count_B#340L, (count_A#335L * count_B#340L) AS provenance#351L]
      +- Project [C#5, A#3, count_A#335L, B#349, count_B#340L]
         +- Join Inner, (C#5 = C#350)
            :- Aggregate [A#3, C#5], [A#3, C#5, count(1) AS count_A#335L]
            :  +- Project [_1#0 AS A#3, _2#1 AS B#4, _3#2 AS C#5]
            :     +- LocalRelation [_1#0, _2#1, _3#2]
            +- Aggregate [B#349, C#350], [B#349, C#350, count(1) AS count_B#340L]
               +- Project [_1#345 AS A#348, _2#346 AS B#349, _3#347 AS C#350]
                  +- LocalRelation [_1#345, _2#346, _3#347]
```


In [9]:
//df.join(df2).withColumn("provenance", df("provenance") * df2("provenance")).drop(df("provenance"), df2("provenance")).show()

### Example $ \pi_{AB}R \bowtie \pi_{BC}R \bigcup \pi_{AC}R \bowtie \pi_{BC}R$  

We can compute the provenance annotation for df4 by summing the provenance annotations from df2 and df3 for each tuple in df4.

We need to use union to avoid double counting the provenance annotations for the tuples that are in both df2 and df3.

We also need to group by the columns and sum the provenance annotations for each tuple in df4.

In [10]:
// Expected results:
// +---+---+---+----------+
// |  A|  B|  C|provenance|
// +---+---+---+----------+
// |  a|  b|  c|         8|
// |  a|  b|  e|        10|
// |  d|  b|  c|        10|
// |  d|  b|  e|        50|
// |  d|  g|  e|         5|
// |  f|  b|  e|         5|
// |  f|  g|  e|         2|
// +---+---+---+----------+

val df4 = df2.union(df3).distinct().orderBy("A", "B", "C")

val df4WithProvenance: DataFrame = df3WithProvenance.union(df2WithProvenance)
                                                    .groupBy("A", "B", "C")
                                                    .agg(sum("provenance").as("provenance"))
                                                    .orderBy("A", "B", "C")

df4WithProvenance.show()



+---+---+---+----------+
|  A|  B|  C|provenance|
+---+---+---+----------+
|  a|  b|  c|         8|
|  a|  b|  e|        10|
|  d|  b|  c|        10|
|  d|  b|  e|        50|
|  d|  g|  e|         5|
|  f|  b|  e|         5|
|  f|  g|  e|         2|
+---+---+---+----------+



df4: Dataset[org.apache.spark.sql.Row] = [A: string, B: string ... 1 more field]
df4WithProvenance: DataFrame = [A: string, B: string ... 2 more fields]

Without provenance
```
== Optimized Logical Plan ==
Sort [A#379 ASC NULLS FIRST, B#380 ASC NULLS FIRST, C#397 ASC NULLS FIRST], true
+- Aggregate [A#379, B#380, C#397], [A#379, B#380, C#397]
   +- Union false, false
      :- Sort [A#379 ASC NULLS FIRST, B#380 ASC NULLS FIRST, C#397 ASC NULLS FIRST], true
      :  +- Project [A#379, B#380, C#397]
      :     +- Join Inner, (B#380 = B#396)
      :        :- LocalRelation [A#379, B#380]
      :        +- LocalRelation [B#396, C#397]
      +- Sort [A#900 ASC NULLS FIRST, B#824 ASC NULLS FIRST, C#902 ASC NULLS FIRST], true
         +- Project [A#900, B#824, C#902]
            +- Join Inner, (C#902 = C#825)
               :- LocalRelation [A#900, C#902]
               +- LocalRelation [B#824, C#825]
```

With provenance
```
== Optimized Logical Plan ==
Sort [A#379 ASC NULLS FIRST, B#824 ASC NULLS FIRST, C#381 ASC NULLS FIRST], true
+- Aggregate [A#379, B#824, C#381], [A#379, B#824, C#381, sum(provenance#841L) AS provenance#909L]
   +- Union false, false
      :- Sort [A#379 ASC NULLS FIRST, B#824 ASC NULLS FIRST, C#381 ASC NULLS FIRST], true
      :  +- Aggregate [A#379, B#824, C#381], [A#379, B#824, C#381, count(1) AS provenance#841L]
      :     +- Project [A#379, B#824, C#381]
      :        +- Join Inner, (C#381 = C#825)
      :           :- LocalRelation [A#379, C#381]
      :           +- LocalRelation [B#824, C#825]
      +- Sort [A#906 ASC NULLS FIRST, B#907 ASC NULLS FIRST, C#397 ASC NULLS FIRST], true
         +- Aggregate [A#906, B#907, C#397], [A#906, B#907, C#397, count(1) AS provenance#413L]
            +- Project [A#906, B#907, C#397]
               +- Join Inner, (B#907 = B#396)
                  :- LocalRelation [A#906, B#907]
                  +- LocalRelation [B#396, C#397]
```

### Example $ \pi_{AC} (\pi_{AB}R \bowtie \pi_{BC}R \bigcup \pi_{AC}R \bowtie \pi_{BC}R)$  

In [11]:
// Expected results:
// +---+---+----------+
// |  A|  C|provenance|
// +---+---+----------+
// |  a|  c|         8|
// |  a|  e|        10|
// |  d|  c|        10|
// |  d|  e|        55|
// |  f|  e|         7|
// +---+---+----------+

val df5 = df4.select("A", "C").distinct().orderBy("A", "C")
df5.show()

val df5WithProvenance: DataFrame = df4WithProvenance.groupBy("A", "C").agg(sum("provenance").as("provenance")).orderBy("A", "C")
df5WithProvenance.show()


+---+---+
|  A|  C|
+---+---+
|  a|  c|
|  a|  e|
|  d|  c|
|  d|  e|
|  f|  e|
+---+---+

+---+---+----------+
|  A|  C|provenance|
+---+---+----------+
|  a|  c|         8|
|  a|  e|        10|
|  d|  c|        10|
|  d|  e|        55|
|  f|  e|         7|
+---+---+----------+



df5: Dataset[org.apache.spark.sql.Row] = [A: string, C: string]
df5WithProvenance: DataFrame = [A: string, C: string ... 1 more field]

Without provenance
```
== Analyzed Logical Plan ==
A: string, C: string
Sort [A#379 ASC NULLS FIRST, C#397 ASC NULLS FIRST], true
+- Deduplicate [A#379, C#397]
   +- Project [A#379, C#397]

      +- Sort [A#379 ASC NULLS FIRST, B#380 ASC NULLS FIRST, C#397 ASC NULLS FIRST], true
         +- Deduplicate [A#379, B#380, C#397]

            +- Union false, false
```

With provenance
```
== Analyzed Logical Plan ==
A: string, C: string, provenance: bigint
Sort [A#379 ASC NULLS FIRST, C#381 ASC NULLS FIRST], true
+- Aggregate [A#379, C#381], [A#379, C#381, sum(provenance#949L) AS provenance#984L]

   +- Sort [A#379 ASC NULLS FIRST, B#824 ASC NULLS FIRST, C#381 ASC NULLS FIRST], true
      +- Aggregate [A#379, B#824, C#381], [A#379, B#824, C#381, sum(provenance#841L) AS provenance#949L]
      
         +- Union false, false
```